Instalação dos pacotes base

In [1]:
%pip install -q -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


Importações

In [2]:
from dotenv import load_dotenv
import os
from groq import Groq
import re


load_dotenv()


True

Modelo que será utilizado 

In [3]:
model = 'llama-3.3-70b-versatile'

Criação do cliente Groq e teste enviando para o modelo uma solicitação

In [4]:
client = Groq(
    api_key=os.environ.get("GROQ_API_KEY"),
)

chat_completion = client.chat.completions.create(
    messages=[
        {
            "role": "user",
            "content": "Explain the importance of fast language models",
        }
    ],
    model=model,
)

print(chat_completion.choices[0].message.content)

Fast language models are crucial in today's technology-driven world, and their importance can be seen in several areas:

1. **Efficient Processing**: Fast language models enable devices to process and respond to natural language inputs in real-time, making them ideal for applications that require quick interactions, such as virtual assistants, chatbots, and voice-controlled systems.
2. **Improved User Experience**: Faster language models lead to more responsive and interactive systems, which enhances the overall user experience. Users can receive instant feedback, answers, and responses, making their interaction with the system more engaging and productive.
3. **Increased Productivity**: Fast language models can automate tasks, such as text summarization, sentiment analysis, and language translation, freeing up human time and increasing productivity. This is particularly useful in industries like customer service, marketing, and content creation.
4. **Better Decision-Making**: Fast lan

Classe Agent, uma abstração do agente. Na rotina de iniciar define algumas propriedades e caso seja passado, insere o system prompt inicial que instrui o modelo a agir. 

Quando a classe é chamada, caso passe uma mensagem é uma entrada do usuário, se não é uma entrada do assistente. 

Implementa a função de execução que realiza as requisições para o modelo. 

In [5]:
class Agent:
    def __init__(self, client: Groq, system: str = "") -> None:
        self.client = client
        self.system = system
        self.messages: list = []
        if self.system:
            self.messages.append({"role": "system", "content": system})

    def __call__(self, message=""):
        if message:
            self.messages.append({"role": "user", "content": message})
        result = self.execute()
        self.messages.append({"role": "assistant", "content": result})
        return result

    def execute(self):
        completion = client.chat.completions.create(
            model=model, messages=self.messages
        )
        return completion.choices[0].message.content

System prompt base que instrui como o agente deve agir. 
Importante ressaltar que é onde são estabelecidos os padrões e formatos de entradas e saídas, como o fluxo de Question > Thought, Action, Observation and Answer. 

Abaixo consta também as funções personalizadas que o agente pode acionar. 

In [6]:
system_prompt = """
You run in a loop of Thought, Action, PAUSE, Observation.
At the end of the loop you output an Answer
Use Thought to describe your thoughts about the question you have been asked.
Use Action to run one of the actions available to you - then return PAUSE.
Observation will be the result of running those actions.

Your available actions are:

calculate:
e.g. calculate: 4 * 7 / 3
Runs a calculation and returns the number - uses Python so be sure to use floating point syntax if necessary

get_planet_mass:
e.g. get_planet_mass: Earth
returns weight of the planet in kg

Example session:

Question: What is the mass of Earth times 2?
Thought: I need to find the mass of Earth
Action: get_planet_mass: Earth
PAUSE 

You will be called again with this:

Observation: 5.972e24

Thought: I need to multiply this by 2
Action: calculate: 5.972e24 * 2
PAUSE

You will be called again with this: 

Observation: 1,1944×10e25

If you have the answer, output it as the Answer.

Answer: The mass of Earth times 2 is 1,1944×10e25.

Now it's your turn:
""".strip()


def calculate(operation: str) -> float:
    return eval(operation)


def get_planet_mass(planet) -> float:
    planet = planet.lower()
    if planet == "earth":
        return 5.972e24
    if planet == "mars":
        return 6.39e23
    if planet == "jupiter":
        return 1.898e27
    if planet == "saturn":
        return 5.683e26
    if planet == "uranus":
        return 8.681e25
    if planet == "neptune":
        return 1.024e26
    if planet == "mercury":
        return 3.285e23
    if planet == "venus":
        return 4.867e24
    return None

Primeiro teste enviando uma pergunta para o agente

In [7]:
neil_tyson = Agent(client=client, system=system_prompt)
result = neil_tyson("What is the mass of Mercury times 5?")

result = neil_tyson()

print(result)

O agente executou a primeira etapa de receber uma pergunta, pensar segundo as instruções recebidas e informar que a ação necessária será a de chamar uma função, então executamos uma chamada ao agente apenas enviando as atualizações como assistente.  

In [8]:
result = get_planet_mass("mercury")
print(result)

next_prompt = "Observation: {}".format(result)
next_prompt

result = neil_tyson(next_prompt)
print(result)

3.285e+23
Thought: Now that I have the mass of Mercury, I need to multiply it by 5 to get the final answer.

Action: calculate: 3.285e+23 * 5
PAUSE


No primeiro momento realizamos a chamada de forma manual para enviar o resultado para o agente. O retorno da função é concatenado em uma string de 'Observation' formatando assim o próximo prompt. 
Em seguida enviamos ao agente esse novo prompt com o resultado da função que foi executada. 

In [9]:

action = re.findall(r"Action: ([a-z_]+): (.+)", result, re.IGNORECASE)
chosen_tool = action[0][0]
arg = action[0][1]

result_tool = eval(f"{chosen_tool}('{arg}')")
next_prompt = f"Observation: {result_tool}"

result = neil_tyson(next_prompt)
result

'Thought: I have now calculated the mass of Mercury times 5, so I can output the final answer.\n\nAnswer: The mass of Mercury times 5 is 1.6425e+24.'

Imprimindo todas as mensagens do assistente e do usuário para ver a jornada completa

In [10]:
for msg in neil_tyson.messages[1:]:
    print(msg['content'])

What is the mass of Mercury times 5?
Thought: To find the mass of Mercury times 5, I first need to find the mass of Mercury. 

Action: get_planet_mass: Mercury
PAUSE

Observation: 3.285e+23
Thought: Now that I have the mass of Mercury, I need to multiply it by 5 to get the final answer.

Action: calculate: 3.285e+23 * 5
PAUSE
Observation: 1.6425e+24
Thought: I have now calculated the mass of Mercury times 5, so I can output the final answer.

Answer: The mass of Mercury times 5 is 1.6425e+24.


Afim de automatizar o fluxo de chamada de funções, o método loop() utiliza um laço de repetição 

In [11]:
def loop(max_iterations=10, query: str = ""):

    agent = Agent(client=client, system=system_prompt)

    tools = ["calculate", "get_planet_mass"]

    next_prompt = query

    i = 0
  
    while i < max_iterations:
        i += 1
        result = agent(next_prompt)
        print(result)

        if "PAUSE" in result and "Action" in result:
            action = re.findall(r"Action: ([a-z_]+): (.+)", result, re.IGNORECASE)
            chosen_tool = action[0][0]
            arg = action[0][1]

            if chosen_tool in tools:
                result_tool = eval(f"{chosen_tool}('{arg}')")
                next_prompt = f"Observation: {result_tool}"

            else:
                next_prompt = "Observation: Tool not found"

            print(next_prompt)
            continue

        if "Answer" in result:
            break


loop(query="What is the mass of Earth plus the mass of Saturn and all of that times 2?")

Thought: To solve this problem, I need to find the mass of Earth and the mass of Saturn, then add them together and multiply the result by 2. First, I'll find the mass of Earth.

Action: get_planet_mass: Earth
PAUSE
Observation: 5.972e+24
Thought: Now that I have the mass of Earth, I need to find the mass of Saturn.

Action: get_planet_mass: Saturn
PAUSE
Observation: 5.683e+26
Thought: I now have the masses of both Earth and Saturn. The next step is to add these two masses together.

Action: calculate: 5.972e+24 + 5.683e+26
PAUSE
Observation: 5.74272e+26
Thought: Now that I have the sum of the masses of Earth and Saturn, I need to multiply this result by 2 to get the final answer.

Action: calculate: 5.74272e+26 * 2
PAUSE
Observation: 1.148544e+27
Thought: I have now calculated the sum of the masses of Earth and Saturn, and then multiplied the result by 2. This is the final step in solving the problem.

Answer: The mass of Earth plus the mass of Saturn and all of that times 2 is 1.1485